#### Objective problem: Với ngân sách khuyến mãi cố định, nên gửi voucher cho nhóm rider nào để tạo ra nhiều incremental trips nhất?

#### Feature:
- Nhóm dữ liệu về thông tin các nhân user: user_id, age, gender, home_borough
- Nhóm dữ liệu về tần suất sử dụng: total_rides, recency_days
- Nhóm dữ liệu hành vi di chuyển: typical_distance, pct_airport
- Nhóm dữ liệu về thời gian sử dụng: pref_time_bucket, weekend_ratio
- Nhóm dữ liệu về thanh toán: pct_flex_payment, pct_tip_rate, avg_fare

#### Synthetic Dataset Design Overview

**1. Dataset Objective**

Mục tiêu của việc sinh dữ liệu synthetic là xây dựng một rider-level dataset mô phỏng dữ liệu khách hàng nhằm phục vụ:

- Customer segmentation để xác định các nhóm rider có hành vi khác nhau.
- Targeting promotion để tìm nhóm khách hàng có khả năng tạo ra incremental trips cao.
- A/B testing.

**2. Unit of Observation**

- Dataset được xây dựng ở cấp độ customer/rider level.
- Mỗi dòng dữ liệu đại diện cho: Một rider duy nhất trong hệ thống ride-hailing platform trong khoảng thời gian quan sát.

Mỗi rider có:

| # | Field | Type | Unit | Ý nghĩa | Ghi chú |
|---|-------|------|------|---------|---------|
| 1 | `user_id` | int64 | code | Mã định danh rider | Khoá chính, duy nhất. `1` → `20000` |
| 2 | `age` | int16 | năm | Tuổi của rider | |
| 3 | `gender` | object | category | Giới tính | `Female` / `Male` / `Other` |
| 4 | `home_borough` | object | category | Khu vực hoạt động chính của rider dựa trên khu vực pickup thường xuyên nhất | |
| 5 | `total_rides` | int32 | chuyến | Tổng số chuyến lịch sử | |
| 6 | `recency_days` | int16 | ngày | Số ngày kể từ chuyến gần nhất | |
| 7 | `typical_distance` | float64 | dặm | Khoảng cách chuyến đi điển hình của rider (median trip distance), đại diện cho loại hình di chuyển thường xuyên | |
| 8 | `route_entropy` | float64 | nat | Độ đa dạng tuyến đi của rider | Entropy Shannon của các cặp zone `PU→DO`. **0** = chỉ đi một tuyến duy nhất; càng cao càng nhiều tuyến khác nhau |
| 9 | `pct_airport` | float64 | 0–1 | Số chuyến airport / Tổng số chuyến | |
| 10 | `pref_time_bucket` | object | category | Khung giờ rider thường sử dụng dịch vụ nhất | morning, midday, evening, late_night |
| 11 | `weekend_ratio` | float64 | 0–1 | Tỷ lệ chuyến vào cuối tuần | |
| 12 | `pct_flex_payment` | float64 | 0–1 | Tỷ lệ chuyến trả bằng Flex Fare | |
| 13 | `pct_tip_rate` | float64 | ratio | Tỷ lệ tip | |
| 14 | `avg_fare` | float64 | USD/trip | Giá cước trung bình mỗi chuyến của rider, phản ánh giá trị giao dịch trung bình | |

**3. Dataset Scale**

| Thành phần | Giá trị |
|---|---|
| Số lượng riders | 20,000 users |
| Đơn vị quan sát | Rider |
| Số feature | 14 features |

**4. Data Sources**

Bộ dữ liệu synthetic được tạo ra dựa trên sự kết hợp của 2 bộ dữ liệu NYC TLC Trip Data và đặc điểm người dùng từ ride-sharing platform.

## Pipeline

```
     NYC TLC
        │
        ▼
  1. Làm sạch + tạo biến
        ▼
  2. Gán archetype cho từng chuyến
        ▼
  3. Sinh 20,000 rider: mỗi rider bốc một số chuyến thật, sao cho phân phối
     hour distribution, distance distribution, location distribution
     trên tập dữ liệu mới tương đương phân phối của TLC
        ▼
  4. Sinh outcome + gán treatment
```

In [2]:
import os
import numpy as np, pandas as pd
from scipy.stats import nbinom, norm, ks_2samp, binomtest
from scipy.optimize import brentq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

B   = r"c:/Users/Linh/Desktop/Growth & Experimentation Project for Ride-Hailing Promotions"
KAG = os.path.expanduser("~/.cache/kagglehub/datasets/adnananam/"
                         "ride-sharing-platform-data/versions/1")
OUTDIR = os.path.join(B, "02. Synthetic_data", "outputs")
os.makedirs(OUTDIR, exist_ok=True)

N_RIDERS = 20_000
SEED     = 42
ALPHA    = 0.35    # do tap trung khau vi archetype cua rider (nho = chuyen mon hoa)

rng = np.random.default_rng(SEED)
print("TLC   :", os.path.isdir(os.path.join(B, "data")))
print("Kaggle:", os.path.isdir(KAG))

TLC   : True
Kaggle: True


## 1. Làm sạch TLC + tạo biến
Đọc file data đã làm sạch ở bước EDA

In [3]:
cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
        "fare_amount", "tip_amount", "payment_type", "RatecodeID",
        "PULocationID", "DOLocationID", "Airport_fee"]

d = pd.read_csv(f"{B}/data/yellow_tripdata_2026.csv", usecols=cols,
                parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"]
                ).rename(columns={"Airport_fee": "airport_fee"})
n_raw = len(d)

d["dur"] = (d.tpep_dropoff_datetime - d.tpep_pickup_datetime).dt.total_seconds() / 60
d = d[(d.fare_amount > 2.5) & (d.dur > 1)].reset_index(drop=True)

# ty le tip chi do duoc tren chuyen the (TLC khong ghi tip tien mat)
d["tip_rate"] = np.where(d.payment_type == 1,
                         np.clip(d.tip_amount / d.fare_amount, 0, 1), np.nan)

d["hour"]    = d.tpep_pickup_datetime.dt.hour.astype(np.int8)
d["is_wknd"] = (d.tpep_pickup_datetime.dt.dayofweek >= 5).astype(np.int8)
d["is_air"]  = (d.RatecodeID.isin([2, 3]) | (d.airport_fee.fillna(0) > 0)).astype(np.int8)
d["is_flex"] = (d.payment_type == 0).astype(np.int8)
d["is_card"] = (d.payment_type == 1).astype(np.int8)

## 2. Gán archetype cho từng chuyến

**16 archetype = 4 khung giờ × 4 nhóm cự ly.**

| | |
|---|---|
| Khung giờ | `morning` 6–10h · `midday` 10–16h · `evening` 16–22h · `late_night` 22–6h |
| Cự ly | `short` < 2 dặm · `medium` 2–5 dặm · `long` 5–10 dặm · `very_long` > 10 dặm |

In [3]:
def time_bucket(hour):
    if 6 <= hour < 10:    return "morning"
    elif 10 <= hour < 16: return "midday"
    elif 16 <= hour < 22: return "evening"
    else:                 return "late_night"

DIST_EDGE  = [2, 5, 10]                                   # short | medium | long | very_long
DIST_LABEL = ["short", "medium", "long", "very_long"]

BUCKET = {h: time_bucket(h) for h in range(24)}
d["time_bucket"]     = d.hour.map(BUCKET)
d["distance_bucket"] = pd.Categorical.from_codes(
    np.searchsorted(DIST_EDGE, d.trip_distance.to_numpy(), side="right"), DIST_LABEL)
d["archetype"]       = d.time_bucket.astype(str) + "_" + d.distance_bucket.astype(str)

ARCH   = sorted(d.archetype.unique())
N_ARCH = len(ARCH)
d["arch"] = d.archetype.map({a: i for i, a in enumerate(ARCH)}).astype(np.int8)
p_arch = d.arch.value_counts(normalize=True).reindex(range(N_ARCH)).to_numpy()

chk = pd.DataFrame({
    "ty trong":    p_arch,
    "dist median": [np.median(d.trip_distance.values[d.arch == k]) for k in range(N_ARCH)],
    "fare median": [np.median(d.fare_amount.values[d.arch == k]) for k in range(N_ARCH)],
    "% san bay":   [d.is_air.values[d.arch == k].mean()*100 for k in range(N_ARCH)],
    "% Flex":      [d.is_flex.values[d.arch == k].mean()*100 for k in range(N_ARCH)],
    "tip rate":    [d.tip_rate.values[(d.arch == k) & (d.is_card == 1)].mean() for k in range(N_ARCH)],
}, index=ARCH).round(3)
print(f"{N_ARCH} archetype = 4 khung gio x {len(DIST_LABEL)} nhom cu ly:")
display(chk)

16 archetype = 4 khung gio x 4 nhom cu ly:


,ty trong,dist median,fare median,% san bay,% Flex,tip rate
evening_long,0.036,6.90,34.50,22.216,36.123,0.200
evening_medium,0.104,2.82,19.80,1.189,31.045,0.231
evening_short,0.204,1.10,10.00,0.147,17.848,0.299
evening_very_long,0.025,16.24,70.00,69.038,10.684,0.176
late_night_long,0.030,6.84,32.27,14.900,55.545,0.202
late_night_medium,0.065,3.05,19.61,1.371,49.456,0.228
late_night_short,0.080,1.20,10.00,0.215,33.174,0.288
late_night_very_long,0.016,14.40,56.20,51.143,29.581,0.168
midday_long,0.031,7.05,35.20,21.076,31.900,0.166
midday_medium,0.078,2.80,21.20,1.011,29.228,0.206


## 3. Sinh 20,000 rider

Mỗi rider được gán với một số chuyến theo archetype từ TLC

Giả định: 
| Cơ chế | Tham số | Giải thích |
|---|---|---|
| Số chuyến | `n ~ NegBinom(3, 0.3)` | Rider càng gắn bó càng đi nhiều |
| Thói quen kiểu chuyến | `w ~ Dirichlet(ALPHA · p_arch · N_ARCH)` | Rider chuyên đi vài kiểu chuyến |

In [ ]:
def rider_skeleton(r):
    """Sinh bo khung chuyen cua 20,000 rider: ai di may chuyen va di kieu gi."""
    u   = norm.cdf(r.standard_normal(N_RIDERS))
    n_i = nbinom.ppf(np.clip(u, 1e-6, 1-1e-6), 3, 0.3).astype(int) + 1     # so chuyen
    w_i = r.dirichlet(ALPHA * p_arch * N_ARCH, size=N_RIDERS)              

    rid = np.repeat(np.arange(N_RIDERS), n_i)                              # chuyen -> rider
    a_f = np.clip((np.cumsum(w_i, 1)[rid]                                  # chuyen -> archetype
                   < r.random(len(rid))[:, None]).sum(1), 0, N_ARCH - 1)
    return rid, a_f

arch  = d.arch.to_numpy()
dist  = d.trip_distance.to_numpy(np.float64)
fare  = d.fare_amount.to_numpy(np.float64)
hour  = d.hour.to_numpy(np.int64)
is_air = d.is_air.to_numpy(np.int8);   is_wknd = d.is_wknd.to_numpy(np.int8)
is_flex = d.is_flex.to_numpy(np.int8); is_card = d.is_card.to_numpy(np.int8)
trate = np.nan_to_num(d.tip_rate.to_numpy(np.float64))
PU = d.PULocationID.to_numpy(np.int64); DO = d.DOLocationID.to_numpy(np.int64)
N_ZONE = 266

# chi muc chuyen that theo archetype -> boc DEU trong tung archetype
ord_a = np.argsort(arch, kind="stable").astype(np.int32)
cnt_a = np.bincount(arch, minlength=N_ARCH)
st_a  = np.concatenate([[0], np.cumsum(cnt_a)])

rid_all, a_all = rider_skeleton(rng)
idx_all = ord_a[st_a[a_all] + (rng.random(len(a_all)) * cnt_a[a_all]).astype(np.int64)].astype(np.int64)
print(f"{N_RIDERS:,} rider / {len(idx_all):,} chuyen "
      f"(trung binh {len(idx_all)/N_RIDERS:.1f} chuyen/rider)")

20,000 rider / 160,634 chuyen (trung binh 8.0 chuyen/rider)


In [5]:
zl = pd.read_csv(f"{B}/data/taxi_zone_lookup.csv")
bmap = dict(zip(zl.LocationID, zl.Borough.fillna("Unknown")))

def aggregate(rid, tr):
    """Tong hop chuyen -> dac trung rider."""
    n   = N_RIDERS
    cnt = np.bincount(rid, minlength=n)
    g   = pd.DataFrame({"r": rid, "dist": tr["dist"], "fare": tr["fare"]})
    med  = g.groupby("r").dist.median().reindex(range(n)).to_numpy()
    avgf = g.groupby("r").fare.mean().reindex(range(n)).to_numpy()
    rmean = lambda x: np.bincount(rid, weights=x.astype(float), minlength=n) / cnt

    card   = tr["is_card"].astype(bool)                 # tip chi do duoc tren chuyen the
    n_card = np.bincount(rid[card], minlength=n)
    s_card = np.bincount(rid[card], weights=tr["trate"][card], minlength=n)

    route = tr["PU"] * 1000 + tr["DO"]                  # entropy Shannon tren cap PU->DO
    c = pd.Series(np.ones(len(rid))).groupby([rid, route]).size()
    p = c / c.groupby(level=0).transform("sum")
    ent = (-(p * np.log(p))).groupby(level=0).sum().reindex(range(n)).fillna(0).to_numpy()

    H = np.zeros((n, 24), np.int32);     np.add.at(H, (rid, tr["hour"]), 1)
    Z = np.zeros((n, N_ZONE), np.int32); np.add.at(Z, (rid, tr["PU"]), 1)

    return pd.DataFrame({
        "total_rides":      cnt,
        "typical_distance": med,
        "route_entropy":    ent,
        "pct_airport":      rmean(tr["is_air"]),
        "weekend_ratio":    rmean(tr["is_wknd"]),
        "pct_flex_payment": rmean(tr["is_flex"]),
        "pct_tip_rate":     np.where(n_card > 0, s_card / np.maximum(n_card, 1), 0.0),
        "avg_fare":         avgf,
        "pref_time_bucket": np.array([BUCKET[h] for h in H.argmax(1)], dtype=object),
        "home_borough":     np.array([bmap.get(int(z), "Unknown") for z in Z.argmax(1)], dtype=object),
    })

In [6]:
trips = dict(dist=dist[idx_all], fare=fare[idx_all], is_air=is_air[idx_all],
             is_wknd=is_wknd[idx_all], is_flex=is_flex[idx_all], is_card=is_card[idx_all],
             trate=trate[idx_all], hour=hour[idx_all], PU=PU[idx_all], DO=DO[idx_all])
riders = aggregate(rid_all, trips)

# ba bien nhan khau hoc lay tu Ride-Sharing Platform Data
ku = pd.read_csv(f"{KAG}/users.csv", parse_dates=["registration_date"])
kr = pd.read_csv(f"{KAG}/rides.csv", parse_dates=["ride_start_time"])
asof = kr.ride_start_time.max().normalize() + pd.Timedelta(days=1)
last = kr.groupby("user_id").ride_start_time.max().reindex(ku.user_id)
rec_days = (asof - last).dt.days.fillna((asof - kr.ride_start_time.min()).days)
Q = np.linspace(0, 1, 1001)
q_age, q_rec = np.quantile(ku.age, Q), np.quantile(rec_days, Q)
pmf_g = ku.gender.value_counts(normalize=True)

riders["age"]          = np.interp(rng.random(N_RIDERS), Q, q_age).round().clip(18, 65).astype(int)
riders["gender"]       = rng.choice(pmf_g.index, N_RIDERS, p=pmf_g.values)
riders["recency_days"] = np.interp(rng.random(N_RIDERS), Q, q_rec).round().clip(0, None).astype(int)

riders["total_rides"] = riders.total_rides.astype(int)
for c in ["typical_distance", "avg_fare"]:
    riders[c] = riders[c].round(3)
for c in ["route_entropy", "pct_airport", "weekend_ratio", "pct_flex_payment", "pct_tip_rate"]:
    riders[c] = riders[c].clip(0, 1 if c != "route_entropy" else None).round(6)

COLS = ["user_id", "age", "gender", "home_borough", "total_rides", "recency_days",
        "typical_distance", "route_entropy", "pct_airport", "pref_time_bucket",
        "weekend_ratio", "pct_flex_payment", "pct_tip_rate", "avg_fare"]
riders.insert(0, "user_id", np.arange(1, N_RIDERS + 1))
riders = riders[COLS]
assert len(COLS) == 14

NUM = ["total_rides", "typical_distance", "route_entropy", "pct_airport", "weekend_ratio",
       "pct_flex_payment", "pct_tip_rate", "avg_fare", "age", "recency_days"]

riders.to_csv(os.path.join(OUTDIR, "customer_features_final.csv"), index=False)
display(riders.head())
display(riders[NUM].describe().round(3))

,user_id,age,gender,home_borough,total_rides,recency_days,typical_distance,route_entropy,pct_airport,pref_time_bucket,weekend_ratio,pct_flex_payment,pct_tip_rate,avg_fare
0,1,24,Female,Manhattan,9,69,2.450,2.197225,0.111111,late_night,0.444444,0.333333,0.102761,19.432
1,2,57,Other,Manhattan,3,16,0.890,1.098612,0.000000,morning,0.333333,0.333333,0.392105,8.873
2,3,25,Female,Manhattan,11,12,0.910,2.271869,0.000000,morning,0.454545,0.090909,0.334748,10.872
3,4,52,Female,Manhattan,12,39,3.095,2.484907,0.000000,midday,0.333333,0.333333,0.201553,20.193
4,5,18,Other,Manhattan,1,32,5.610,0.000000,0.000000,morning,0.000000,1.000000,0.000000,36.170


,total_rides,typical_distance,route_entropy,pct_airport,weekend_ratio,pct_flex_payment,pct_tip_rate,avg_fare,age,recency_days
count,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000,20000.000
mean,8.032,2.637,1.884,0.068,0.294,0.265,0.240,21.314,40.998,56.097
std,4.868,2.262,0.663,0.124,0.204,0.201,0.080,9.127,13.875,54.437
min,1.000,0.010,0.000,0.000,0.000,0.000,0.000,3.000,18.000,0.000
25%,4.000,1.395,1.386,0.000,0.167,0.125,0.203,15.177,29.000,16.000
50%,7.000,1.940,1.946,0.000,0.286,0.250,0.248,19.551,41.000,39.000
75%,11.000,2.895,2.305,0.111,0.400,0.375,0.286,25.309,53.000,79.000
max,44.000,30.310,3.761,1.000,1.000,1.000,1.000,146.500,65.000,277.000


## 4. Sinh outcome + gán treatment

Mỗi rider có **hai** outcome: `Y0` (không voucher) và `Y1` (có voucher), đo bằng số chuyến trong 30 ngày. 

Dữ liệu xuất ra hai nhánh:

- `T_rct` — gán ngẫu nhiên 50/50 không confounding.
- `T_obs` — có confounder 

In [ ]:
BASELINE_MEAN = 8.0      # E[Y0] muc tieu, chuyen / 30 ngay
NB_DISPERSION = 4.0      # r cua negative binomial; nho -> duoi day
ZERO_TARGET   = 0.20     # ty le rider ngu dong (Y0 = 0)
WAKE_RATE     = 0.25     # voucher danh thuc 25% nguoi ngu dong

B_PRIOR, B_URBAN, B_PRICE = 0.35, 0.20, 0.10

TAU_BASE   = 1.5
D_SHORT    = 1.0     # tren z(-typical_distance)
D_MIDDAY   = 0.6     # co dinh cho rider troi khung midday
P_MORNING  = 0.5     # tru cho rider troi khung morning
P_AIRPORT  = 0.5     # tren z(pct_airport)
D_PRICE    = 0.6
TAU_NOISE  = 0.25

TARGETING = {"prior": 0.85, "urban": 0.60, "price": 0.45}   # nhanh OBS

df = riders.copy()

def zs(x):
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std()

# Ba confounder
df["is_urban"] = df.home_borough.eq("Manhattan").astype(np.int8)
df["z_prior"]  = zs(np.log1p(df.total_rides))
df["z_price"]  = zs(-df.pct_tip_rate)          # tip thap -> nhay gia
df["z_urban"]  = zs(df.is_urban)
df["z_air"]    = zs(df.pct_airport)

print(f"is_urban = {df.is_urban.mean():.1%}")
display(df[["z_prior", "z_price", "z_urban", "z_air"]].describe().round(3))

is_urban = 81.1%


,z_prior,z_price,z_urban,z_air
count,20000.000,20000.000,20000.000,20000.000
mean,0.000,-0.000,0.000,-0.000
std,1.000,1.000,1.000,1.000
min,-2.475,-9.476,-2.070,-0.546
25%,-0.812,-0.577,0.483,-0.546
50%,0.041,-0.092,0.483,-0.546
75%,0.777,0.463,0.483,0.352
max,3.176,2.996,0.483,7.534


### 4.1 Hai outcome tiềm năng

In [9]:
from scipy.stats import nbinom as NB

rng_ab = np.random.default_rng(SEED)

df["z_dist"] = zs(-df.typical_distance)          # NGAN = duong = co gian cao
_mid = (df.pref_time_bucket == "midday").astype(float)
_mor = (df.pref_time_bucket == "morning").astype(float)

tau = (TAU_BASE
       + D_SHORT   * df.z_dist
       + D_MIDDAY  * _mid
       - P_MORNING * _mor
       - P_AIRPORT * df.z_air
       + D_PRICE   * df.z_price
       + TAU_NOISE * rng_ab.standard_normal(len(df)))
df["tau_true"] = np.clip(tau, 0, None)

mu0 = BASELINE_MEAN * np.exp(B_PRIOR*df.z_prior + B_URBAN*df.z_urban + B_PRICE*df.z_price)
mu1 = mu0 + df.tau_true

lin0 = -(0.90*df.z_prior + 0.30*df.z_urban).to_numpy()
def p_zero(c):
    pi = 1/(1+np.exp(-(c+lin0)))
    p_nb0 = (NB_DISPERSION/(NB_DISPERSION+mu0.to_numpy()))**NB_DISPERSION
    return (pi + (1-pi)*p_nb0).mean()
c0  = brentq(lambda c: p_zero(c) - ZERO_TARGET, -20, 20)
pi0 = 1/(1+np.exp(-(c0+lin0)))
pi1 = pi0 * (1 - WAKE_RATE)                       # voucher danh thuc nguoi ngu dong

def draw(mu, pi):
    p = NB_DISPERSION/(NB_DISPERSION+np.asarray(mu, float))
    y = NB.rvs(NB_DISPERSION, p, random_state=rng_ab.integers(1e9))
    return np.where(rng_ab.random(len(y)) < pi, 0, y).astype(np.int32)

df["Y0"] = draw(mu0, pi0)
df["Y1"] = np.maximum(draw(mu1, pi1), 0)

df["tau_shift"] = df.tau_true
df["tau_true"]  = (1 - pi1) * mu1 - (1 - pi0) * mu0
ATE_TRUE = df.tau_true.mean()

print(f"tau tren tham so mean    : {df.tau_shift.mean():.4f}")
print(f"tau tren outcome quan sat: {ATE_TRUE:.4f}")
print(f"P(Y0 = 0)   = {(df.Y0==0).mean():.4f}")
print(f"E[Y0]       = {df.Y0.mean():.3f} chuyen / 30 ngay")
print(f"tau: min {df.tau_true.min():.2f} | p25 {df.tau_true.quantile(.25):.2f} "
      f"| median {df.tau_true.median():.2f} | p75 {df.tau_true.quantile(.75):.2f} "
      f"| max {df.tau_true.max():.2f}")

tau tren tham so mean    : 1.7491
tau tren outcome quan sat: 1.8344
P(Y0 = 0)   = 0.2019
E[Y0]       = 7.387 chuyen / 30 ngay
tau: min 0.11 | p25 1.30 | median 1.90 | p75 2.42 | max 4.59


### 4.2 Gán treatment cho hai nhánh

In [10]:
CLU = ["route_entropy", "pct_tip_rate", "pct_flex_payment",
       "pct_airport", "total_rides", "weekend_ratio"]
Xc = StandardScaler().fit_transform(df[CLU].to_numpy(float))
df["cluster"] = KMeans(4, n_init=10, random_state=SEED).fit_predict(Xc)

zc_ = (df[CLU] - df[CLU].mean()) / df[CLU].std()
zc_["cluster"] = df.cluster
prof = zc_.groupby("cluster").mean()
prof.columns = [f"z_{c}" for c in prof.columns]
display(prof[["z_pct_tip_rate", "z_total_rides", "z_pct_flex_payment", "z_pct_airport"]].round(2))

df["blk_freq"]  = pd.qcut(df.total_rides,  5, labels=False, duplicates="drop")
df["blk_price"] = pd.qcut(df.pct_tip_rate, 5, labels=False, duplicates="drop")
df["block_id"]  = df.blk_freq.astype(str) + "_" + df.blk_price.astype(str)

TREAT_RATE = 0.50
df["T_rct"] = 0
for _, g in df.groupby("block_id"):        # boc 50% TRONG TUNG KHOI
    k = int(round(len(g) * TREAT_RATE))
    df.loc[rng_ab.choice(g.index, k, replace=False), "T_rct"] = 1
N_TREATED = int(df.T_rct.sum())

lin = (TARGETING["prior"]*df.z_prior + TARGETING["urban"]*df.z_urban
       + TARGETING["price"]*df.z_price).to_numpy()
c1 = brentq(lambda c: (1/(1+np.exp(-(c+lin)))).sum() - N_TREATED, -20, 20)
df["propensity_true"] = 1/(1+np.exp(-(c1+lin)))
df["T_obs"] = (rng_ab.random(len(df)) < df.propensity_true).astype(np.int8)

df["Y_rct"] = np.where(df.T_rct == 1, df.Y1, df.Y0).astype(np.int32)
df["Y_obs"] = np.where(df.T_obs == 1, df.Y1, df.Y0).astype(np.int32)
print(f"T_rct = {N_TREATED:,} ({df.T_rct.mean():.1%}) | T_obs = {int(df.T_obs.sum()):,} ({df.T_obs.mean():.1%})")

,z_pct_tip_rate,z_total_rides,z_pct_flex_payment,z_pct_airport
cluster,,,,
0,-0.25,-0.43,-0.43,2.04
1,-0.87,-0.69,1.26,-0.39
2,0.10,0.99,0.05,-0.11
3,0.38,-0.62,-0.47,-0.45


T_rct = 10,002 (50.0%) | T_obs = 9,999 (50.0%)
